# Phase 1 — NetworkX Prototype

Validates the structural algorithms (cycles, communities, centrality) on a hand-built graph before promoting to Neo4j Cypher/GDS (see `02_TRD.md` §4 and `06_Implementation_Plan.md` Phase 1).

In [ ]:
import networkx as nx

# Build the hero ring: A->B->C->D->E->F->A plus a clean edge N1->N2
G = nx.DiGraph()
hero = ['A', 'B', 'C', 'D', 'E', 'F']
for i in range(len(hero)):
    G.add_edge(hero[i], hero[(i + 1) % len(hero)], amount=500000 - i * 50000)
G.add_edge('N1', 'N2', amount=120000)

print('Nodes:', G.number_of_nodes(), 'Edges:', G.number_of_edges())

In [ ]:
# 1. Cycle detection (TRD §4.1) — simple_cycles finds the planted 6-cycle
cycles = list(nx.simple_cycles(G))
print(f'Cycles found: {len(cycles)}')
for c in sorted(cycles, key=len, reverse=True)[:5]:
    total = sum(G[u][v]['amount'] for u, v in zip(c, c[1:] + [c[0]]))
    print(f'  {len(c)}-hop: {c}  total=₹{total:,}')

In [ ]:
# 2. Weakly connected components (TRD §4.2)
for i, comp in enumerate(nx.weakly_connected_components(G)):
    print(f'Component {i}: {sorted(comp)}')

In [ ]:
# 3. Centrality features (TRD §4.3)
pr = nx.pagerank(G)
bc = nx.betweenness_centrality(G)
for n in sorted(pr, key=pr.get, reverse=True):
    print(f'{n:>2}  PageRank={pr[n]:.4f}  Betweenness={bc[n]:.4f}')

In [ ]:
# 4. Louvain communities on the undirected projection
U = G.to_undirected()
try:
    import networkx.algorithms.community as nx_comm
    comms = nx_comm.louvain_communities(U, seed=42)
    for i, c in enumerate(comms):
        print(f'Community {i}: {sorted(c)}')
except AttributeError:
    print('louvain_communities requires networkx >= 2.7 (installed: ', nx.__version__, ')')

## Promotion to production

The same logic runs against Neo4j via:
- Cycle detection: `MATCH p=(c)-[:TRANSACTS_WITH*3..6]->(c)` → materialized `:FraudCycle` nodes
- WCC/Louvain: `gds.louvain.write` → `:FraudCommunity` + `community_id` write-back
- Centrality: `gds.pageRank.stream` / `gds.betweenness.stream` → write-back fields

Endpoints: `GET /api/v1/analytics/loops`, `POST /api/v1/analytics/structural`, `GET /api/v1/analytics/centrality/{cin}`